In [ ]:
%pip install transformers
%pip install datasets

%pip install torch torchvision torchaudio
%pip install transformers datasets
%pip install numpy pandas scikit-learn
%pip install "accelerate>=1.1.0"
%pip install scikit-learn

In [ ]:
from datasets import load_dataset

# Load goEmotions Dataset and get Sentiment Labels
goEmotionsDataset = load_dataset('go_emotions')
sentimentLabels = goEmotionsDataset['train'].features['labels'].feature.names

In [ ]:
from transformers import RobertaForSequenceClassification

model_name = 'cardiffnlp/twitter-roberta-base'

# Download basic roBERTa model
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(sentimentLabels),
    problem_type='multi_label_classification',
    ignore_mismatched_sizes=True
)

In [ ]:
import numpy as np

# Convert dataset sentiment keys into one hot encoding
def one_hot_encode(example):
    vector = np.zeros(28, dtype=np.float32)

    for label_index in example['labels']:
        vector[label_index] = 1.0

    return {'labels': vector}

dataset = goEmotionsDataset.map(one_hot_encode)

print("Original ID:", goEmotionsDataset['train'][0]['labels'])
print("New Vector:", dataset['train'][0]['labels'])

In [ ]:
from transformers import RobertaTokenizer
from datasets import Sequence, Value

# Tokenize and format for PyTorch
def tokenize_text(rows):
    return tokenizer(rows['text'], padding='max_length', truncation=True, max_length=128)

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
tokenized_dataset = dataset.map(tokenize_text, batched=True)
tokenized_dataset = tokenized_dataset.cast_column("labels", Sequence(Value("float32")))
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Set up F1 Score
def compute_metrics(eval_pred):
    threshold = 0.3

    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > threshold).astype(int)
    return {'f1': f1_score(labels, predictions, average='weighted')}


In [ ]:
import torch
import numpy as np

# 1. Extract labels from the tokenized training dataset
train_labels = np.array(tokenized_dataset['train']['labels'])

# 2. Calculate count of positives (1s) and negatives (0s) per class
pos_counts = np.sum(train_labels, axis=0)
neg_counts = len(train_labels) - pos_counts

# 3. Calculate pos_weight: (Number of Negatives) / (Number of Positives)
pos_weight_calc = neg_counts / (pos_counts + 1e-6)

# 4. Convert to Tensor for PyTorch
pos_weight_tensor = torch.tensor(pos_weight_calc, dtype=torch.float)

print("Calculated Class Weights (First 5):", pos_weight_tensor[:5])
print("Weight for rare classes (e.g., index 13):", pos_weight_tensor[13])

In [ ]:
from torch import topk
from transformers import pipeline

#Pipeline
model_path = "../.saved_models/UpdatedV2TwitterRoBERTa"

emotion_classifier = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,
    truncation=True,
    max_length=512,
    topk=None
)

# Test
result = emotion_classifier("I am sorry for your loss", top_k=10)
for prediction in result:
    sentiment_id = int(prediction['label'].replace("LABEL_", ""))
    sentiment_name = goEmotionsDataset['train'].features['labels'].feature.names[sentiment_id]

    print(f"{sentiment_name}, Score: {prediction['score']}")

In [ ]:
from sklearn.metrics import f1_score
from transformers import RobertaForSequenceClassification, Trainer


model = RobertaForSequenceClassification.from_pretrained(model_path)
trainer = Trainer(model=model)

def optimize_thresholds(trainer, dataset):
    # 1. Get raw logits from the model
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    true_labels = predictions.label_ids
    
    # 2. Convert to probabilities
    probs = 1 / (1 + np.exp(-logits))
    
    best_thresholds = []
    best_f1s = []
    
    # 3. Iterate over all 28 labels
    for i in range(28):
        best_t = 0.5
        best_f1 = 0.0
        
        # Check thresholds from 0.1 to 0.9
        for t in np.arange(0.1, 0.95, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            # Calculate F1 for this specific class
            score = f1_score(true_labels[:, i], preds, zero_division=0)
            
            if score > best_f1:
                best_f1 = score
                best_t = t
                
        best_thresholds.append(best_t)
        best_f1s.append(best_f1)
        
    return np.array(best_thresholds)

# Run optimization
optimal_thresholds = optimize_thresholds(trainer, tokenized_dataset['validation'])

print("Optimal Thresholds per class:")
for name, thresh in zip(sentimentLabels, optimal_thresholds):
    print(f"{name}: {thresh:.3f}")

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset, Sequence, Value
from sklearn.metrics import classification_report

# Evaluate on Reddit test CSV
csv_path = "../Test Dataset Creation/BERT_labeled_reddit_comments.csv"
df = pd.read_csv(csv_path).dropna(subset=["comment_body", "predicted_emotion"])

# Map emotion string -> 28-d one-hot vector
label_to_idx = {name: i for i, name in enumerate(sentimentLabels)}

def emotion_to_vector(emotion: str):
    vec = np.zeros(len(sentimentLabels), dtype=np.float32)
    idx = label_to_idx.get(emotion)
    if idx is not None:
        vec[idx] = 1.0
    return vec

# Ensure labels are stored as plain lists for HF Datasets
df["labels"] = df["predicted_emotion"].apply(lambda e: emotion_to_vector(e).tolist())

reddit_dataset = Dataset.from_pandas(
    df[["comment_body", "labels"]].rename(columns={"comment_body": "text"}),
    preserve_index=False,
)

reddit_dataset = reddit_dataset.cast_column("labels", Sequence(Value("float32")))

reddit_tokenized = reddit_dataset.map(tokenize_text, batched=True)
reddit_tokenized = reddit_tokenized.cast_column("labels", Sequence(Value("float32")))
reddit_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

predictions = trainer.predict(reddit_tokenized)
logits = predictions.predictions
true_labels = predictions.label_ids

sigmoid_outputs = 1 / (1 + np.exp(-logits))
predicted_labels = (sigmoid_outputs > optimal_thresholds).astype(int)

print("Reddit CSV classification report (28 emotions):")
print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=sentimentLabels,
        zero_division=0,
    )
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

report_dict = classification_report(true_labels, predicted_labels, 
                                   target_names=sentimentLabels, 
                                   output_dict=True, zero_division=0)

# Convert to DataFrame and plot
report_df = pd.DataFrame(report_dict).iloc[:-1, :].T
plt.figure(figsize=(10, 8))
sns.heatmap(report_df, annot=True, cmap='RdYlGn')
plt.title("Classification Report Heatmap")
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# 1. Define your Map
sentimentMap = {
  "positive": ["admiration", "amusement", "approval", "caring", "desire", "excitement", "gratitude", "joy", "love", "optimism", "pride", "relief"],
  "negative": ["anger", "annoyance", "disappointment", "disapproval", "disgust", "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"],
  "neutral": ["neutral", "realization", "surprise", "curiosity", "confusion"]
}

# 2. Pre-calculate indices to speed up processing
pos_indices = [sentimentLabels.index(e) for e in sentimentMap['positive'] if e in sentimentLabels]
neg_indices = [sentimentLabels.index(e) for e in sentimentMap['negative'] if e in sentimentLabels]
neu_indices = [sentimentLabels.index(e) for e in sentimentMap['neutral'] if e in sentimentLabels]

def get_bucket_preds(matrix):
    """
    Collapses 28-emotion vectors into 3-bucket vectors (Pos, Neg, Neu).
    Returns a list of single labels (0, 1, or 2).
    """
    bucket_preds = []
    for row in matrix:
        # We sum the scores/probabilities for each bucket
        pos_score = np.sum(row[pos_indices])
        neg_score = np.sum(row[neg_indices])
        neu_score = np.sum(row[neu_indices])

        # 0=Positive, 1=Negative, 2=Neutral (matches target_names order below)
        bucket_preds.append(np.argmax([pos_score, neg_score, neu_score]))
    return bucket_preds

probs = 1 / (1 + np.exp(-logits))

# 4. Generate Bucket Predictions
y_true_buckets = get_bucket_preds(true_labels)
y_pred_buckets = get_bucket_preds(probs)

# 5. Print Report
print("RoBERTa Baseline (Mapped to 3 Buckets):")
print(classification_report(
    y_true_buckets,
    y_pred_buckets,
    target_names=['Positive', 'Negative', 'Neutral']
))

In [ ]:
import time

print("Starting latency test (Reddit CSV)...")
start_time = time.time()

_ = trainer.predict(reddit_tokenized)

end_time = time.time()
total_time = end_time - start_time
num_samples = len(reddit_dataset)

print(f"Processed {num_samples} samples in {total_time:.2f} seconds.")
print(f"Average Latency: {(total_time / num_samples) * 1000:.2f} milliseconds per comment.")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true_buckets, y_pred_buckets)
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Positive', 'Negative', 'Neutral'])

fig, ax = plt.subplots(figsize=(8, 6))
cmd.plot(cmap='Blues', ax=ax)
plt.title("RoBERTa Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# 1. Generate the raw multi-label confusion matrices
mcm = multilabel_confusion_matrix(true_labels, predicted_labels)

# 2. Set up the grid
fig, axes = plt.subplots(7, 4, figsize=(16, 24))
axes = axes.ravel() 

# 3. Loop through and plot each emotion's normalized matrix
for i, (emotion_name, matrix) in enumerate(zip(sentimentLabels, mcm)):
    row_sums = matrix.sum(axis=1)
    matrix_normalized = np.divide(
        matrix.astype('float'), 
        row_sums[:, np.newaxis], 
        out=np.zeros_like(matrix, dtype=float), 
        where=row_sums[:, np.newaxis]!=0
    )
    
    # Create the display object
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix_normalized, display_labels=['Not', 'Is'])
    
    # Plot it (switched values_format to '.2f' for decimals)
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False, values_format='.2f')
    
    axes[i].set_title(f"Emotion: {emotion_name.capitalize()}", fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')

# 4. Adjust spacing and show
plt.tight_layout()
plt.show()

In [ ]:
# Ensure we are looking at the 3-bucket predictions
# 0 = Positive, 1 = Negative, 2 = Neutral

false_positives = []

# Loop through predictions and match them back to the original dataframe
for i, (true_label, pred_label) in enumerate(zip(y_true_buckets, y_pred_buckets)):
    
    # Check if True was Negative (1) but Model Predicted Positive (0)
    if true_label == 1 and pred_label == 0:
        # Get the original text from the dataframe created in Cell 9
        comment = df.iloc[i]['comment_body']
        
        # Get the raw predicted probabilities for context
        pos_score = np.sum(probs[i][pos_indices])
        neg_score = np.sum(probs[i][neg_indices])
        
        false_positives.append((comment, pos_score, neg_score))

print(f"Found {len(false_positives)} instances where True=Negative but Predicted=Positive.\n")
print("-" * 50)

# Print the first 15 examples
for i, (comment, p_score, n_score) in enumerate(false_positives[15:40]):
    print(f"Comment {i+1}: {comment}")
    print(f"-> Model Scores | Positive: {p_score:.2f} | Negative: {n_score:.2f}")
    print("-" * 50)